# Deutsch–Jozsa Algorithm — Teaching Notebook

Deutsch–Jozsa generalizes Deutsch's one-bit problem to an $n$-bit Boolean function under a strong promise: the function is either constant or perfectly balanced.

This notebook is designed to be **self-explanatory for classroom teaching**. It moves from the problem statement and mathematics to quantum circuits, Qiskit implementation, interpretation, limitations, and exercises.

### Learning objectives
- understand promise problems
- derive the $n$-qubit interference pattern
- build constant and balanced phase oracles
- prove why $|0^n
angle$ identifies constant functions
- compare deterministic classical and quantum query complexity

## 0. Installation
Run this only if Qiskit is not already installed.

In [1]:
%pip install -q qiskit qiskit-aer matplotlib numpy

Note: you may need to restart the kernel to use updated packages.


## 1. Imports and helper functions

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction
from math import gcd, pi

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator

backend = AerSimulator()

def run_counts(qc, shots=2048):
    compiled = transpile(qc, backend)
    return backend.run(compiled, shots=shots).result().get_counts()

def plot_counts(counts, title='Measurement counts'):
    keys = sorted(counts)
    vals = [counts[k] for k in keys]
    plt.figure(figsize=(8,4))
    plt.bar(keys, vals)
    plt.xlabel('bit string')
    plt.ylabel('counts')
    plt.title(title)
    plt.xticks(rotation=45)
    plt.show()

/var/folders/ch/1rsqd02j4xjgnwbds4jk8yfm0000gn/T/ipykernel_8581/2428700715.py:6: DeprecationWarning: Using Qiskit with Python 3.9 is deprecated as of the 2.1.0 release. Support for running Qiskit with Python 3.9 will be removed in the 2.3.0 release, which coincides with when Python 3.9 goes end of life.
  from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile


# Part I — Problem and promise
We are given

$$f:\{0,1\}^n\to\{0,1\}$$

with the promise that $f$ is either:

- **constant:** the same output for all $2^n$ inputs;
- **balanced:** output 0 on exactly half of the inputs and 1 on exactly half.

Without the promise there are many other possibilities and the Deutsch–Jozsa conclusion would not be valid.

## 2. Classical deterministic cost
To prove that a function is constant, a deterministic classical algorithm may need

$$2^{n-1}+1$$

queries in the worst case: the first $2^{n-1}$ outputs could all be identical, and only the next query proves balance.

# Part II — Quantum state preparation
Start with

$$|0\rangle^{\otimes n}|1\rangle.$$

After Hadamards,

$$\frac{1}{\sqrt{2^n}}\sum_{x=0}^{2^n-1}|x\rangle|-\rangle.$$

The oracle performs phase kickback:

$$|x\rangle|-\rangle\mapsto(-1)^{f(x)}|x\rangle|-\rangle.$$

Therefore the input register becomes

$$\frac1{\sqrt{2^n}}\sum_x(-1)^{f(x)}|x\rangle.$$

## 3. Why the all-zero output matters
After $H^{\otimes n}$, the amplitude of $|0^n\rangle$ is

$$\frac1{2^n}\sum_x(-1)^{f(x)}.$$

- Constant 0: every term is $+1$, amplitude $=1$.
- Constant 1: every term is $-1$, amplitude $=-1$; probability is still 1.
- Balanced: half $+1$, half $-1$, sum $=0$.

Hence measuring $0^n$ means **constant**; any other bit string means **balanced**.

# Part III — Oracle construction
A convenient balanced example is

$$f(x)=a\cdot x\pmod2$$

for a nonzero bit mask $a$. Its phase oracle can be implemented by $Z$ gates on the selected input qubits.

In [3]:
def dj_phase_oracle(n, kind='constant0', mask=None):
    qc = QuantumCircuit(n, name='U_f phase')
    if kind == 'constant0':
        pass
    elif kind == 'constant1':
        qc.global_phase += np.pi
    elif kind == 'balanced':
        if mask is None:
            mask = [1] + [0]*(n-1)
        if not any(mask):
            raise ValueError('balanced mask must be nonzero')
        for q,b in enumerate(mask):
            if b:
                qc.z(q)
    else:
        raise ValueError
    return qc

print(dj_phase_oracle(4,'balanced',[1,0,1,1]).draw())

     ┌───┐
q_0: ┤ Z ├
     └───┘
q_1: ─────
     ┌───┐
q_2: ┤ Z ├
     ├───┤
q_3: ┤ Z ├
     └───┘


## 4. Phase-oracle version of Deutsch–Jozsa
The ancilla can be conceptually eliminated once we recognize that its only role is to convert $f(x)$ into $(-1)^{f(x)}$.

In [4]:
def deutsch_jozsa(n, kind='constant0', mask=None):
    qc = QuantumCircuit(n,n)
    qc.h(range(n))
    qc.compose(dj_phase_oracle(n,kind,mask), inplace=True)
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc

qc = deutsch_jozsa(4,'balanced',[1,0,1,1])
print(qc.draw())
print(run_counts(qc))

     ┌───┐┌───┐┌───┐   ┌─┐      
q_0: ┤ H ├┤ Z ├┤ H ├───┤M├──────
     ├───┤├───┤└───┘┌─┐└╥┘      
q_1: ┤ H ├┤ H ├─────┤M├─╫───────
     ├───┤├───┤┌───┐└╥┘ ║ ┌─┐   
q_2: ┤ H ├┤ Z ├┤ H ├─╫──╫─┤M├───
     ├───┤├───┤├───┤ ║  ║ └╥┘┌─┐
q_3: ┤ H ├┤ Z ├┤ H ├─╫──╫──╫─┤M├
     └───┘└───┘└───┘ ║  ║  ║ └╥┘
c: 4/════════════════╩══╩══╩══╩═
                     1  0  2  3 
{'1101': 2048}


## 5. Test constant and balanced cases

In [5]:
tests = [
    ('constant0',None),
    ('constant1',None),
    ('balanced',[1,0,0,0]),
    ('balanced',[1,1,0,1]),
]
for kind,mask in tests:
    counts=run_counts(deutsch_jozsa(4,kind,mask))
    print(kind,mask,counts)

constant0 None {'0000': 2048}
constant1 None {'0000': 2048}
balanced [1, 0, 0, 0] {'0001': 2048}
balanced [1, 1, 0, 1] {'1011': 2048}


# Part IV — Full ancilla-oracle version
For teaching, it is useful to see the original oracle model

$$U_f|x,y\rangle=|x,y\oplus f(x)\rangle.$$

For $f(x)=a\cdot x$, use CNOTs from every selected input qubit to the ancilla.

In [6]:
def dj_with_ancilla(n, mask):
    qc = QuantumCircuit(n+1,n)
    anc=n
    qc.x(anc)
    qc.h(range(n+1))
    for q,b in enumerate(mask):
        if b: qc.cx(q,anc)
    qc.h(range(n))
    qc.measure(range(n),range(n))
    return qc

print(dj_with_ancilla(3,[1,1,0]).draw())
print(run_counts(dj_with_ancilla(3,[1,1,0])))

     ┌───┐          ┌───┐          ┌─┐   
q_0: ┤ H ├───────■──┤ H ├──────────┤M├───
     ├───┤       │  └───┘     ┌───┐└╥┘┌─┐
q_1: ┤ H ├───────┼─────────■──┤ H ├─╫─┤M├
     ├───┤┌───┐  │   ┌─┐   │  └───┘ ║ └╥┘
q_2: ┤ H ├┤ H ├──┼───┤M├───┼────────╫──╫─
     ├───┤├───┤┌─┴─┐ └╥┘ ┌─┴─┐      ║  ║ 
q_3: ┤ X ├┤ H ├┤ X ├──╫──┤ X ├──────╫──╫─
     └───┘└───┘└───┘  ║  └───┘      ║  ║ 
c: 3/═════════════════╩═════════════╩══╩═
                      2             0  1 
{'011': 2048}


# Part V — Query complexity and practical interpretation
Deutsch–Jozsa gives a one-query exact quantum algorithm for this **promised** problem. However, randomized classical algorithms can distinguish constant from balanced with high confidence using only a small number of queries. Therefore the algorithm is historically important as an early demonstration of quantum advantage, but not a practical exponential speedup for a naturally occurring task.

# Part VI — Interference picture
The first Hadamard layer spreads amplitude uniformly. The oracle changes signs. The second Hadamard layer performs a Walsh–Hadamard transform that causes sign patterns to interfere. The all-zero component is exactly the average of $(-1)^{f(x)}$.

In [7]:
def walsh_spectrum_from_truth_table(values):
    v=np.array([(-1)**int(b) for b in values],dtype=float)
    n=int(np.log2(len(v)))
    H=np.array([[1.0]])
    for _ in range(n):
        H=np.kron(H, np.array([[1,1],[1,-1]])/np.sqrt(2))
    return H@v/np.sqrt(len(v))

print('constant:', walsh_spectrum_from_truth_table([0]*8))
print('balanced:', np.round(walsh_spectrum_from_truth_table([0,1,0,1,0,1,0,1]),3))

constant: [1. 0. 0. 0. 0. 0. 0. 0.]
balanced: [0. 1. 0. 0. 0. 0. 0. 0.]


# Part VII — Common misconceptions
- The promise is essential.
- A nonzero measurement does **not** identify the entire function; it only proves that the function is not constant under the promise.
- The algorithm's mathematical transform is the Walsh–Hadamard transform, not the QFT.
- Query complexity does not automatically equal total runtime complexity.

# Part VIII — Exercises
1. Derive the amplitude of an arbitrary output $|z\rangle$.
2. Why does constant 1 still lead to measurement $0^n$?
3. Construct a 5-qubit balanced oracle with mask 10110.
4. What breaks if the function is 25% ones and 75% zeros?
5. Compare deterministic, randomized, and quantum query complexity qualitatively.

# Part IX — Solutions
For output $z$,

$$A(z)=\frac1{2^n}\sum_x(-1)^{f(x)+x\cdot z}.$$

For constant 1, the entire state acquires a global minus sign before the last Hadamards; global phase does not affect measurement probabilities. For a non-promised function, the probability of $0^n$ is generally neither 0 nor 1, so the deterministic classification rule fails.

# Compact reference
$$|0^n\rangle\xrightarrow{H^{\otimes n}}\frac1{\sqrt{2^n}}\sum_x|x\rangle\xrightarrow{U_f}\frac1{\sqrt{2^n}}\sum_x(-1)^{f(x)}|x\rangle\xrightarrow{H^{\otimes n}}\cdots$$

Measure:

$$0^n\Rightarrow\text{constant},\qquad z\neq0^n\Rightarrow\text{balanced}.